# API FastAPI — Démo

Ce notebook exerce l'API sans lancer de serveur séparé : `TestClient` (fourni par FastAPI, basé sur `httpx`) appelle directement l'application Python en mémoire. Pratique pour explorer route par route pendant la démo.

In [5]:
import sys
sys.path.append("../api")

from fastapi.testclient import TestClient
import pandas as pd

from main import app

client = TestClient(app)

## 1. Routes Système

In [6]:
reponse = client.get("/info")
print(reponse.status_code)
reponse.json()

200


{'nom': 'CinéData API',
 'version': '1.0.0',
 'description': "API d'exposition des données du projet CinéData (films, genres, équipe, classements)."}

In [7]:
reponse = client.get("/health")
print(reponse.status_code)
reponse.json()

200


{'status': 'ok'}

## 2. Liste paginée des films

In [8]:
reponse = client.get("/movie/")
donnees = reponse.json()

print("Pagination :", donnees["pagination"])
pd.DataFrame(donnees["resultats"])

Pagination : {'page': 1, 'par_page': 10, 'total': 1973, 'total_pages': 198}


,id,title,annee
0,11,Star Wars,1977
1,12,Finding Nemo,2003
2,13,Forrest Gump,1994
3,14,American Beauty,1999
4,18,The Fifth Element,1997
5,22,Pirates of the Caribbean: The Curse of the Bla...,2003
6,24,Kill Bill: Vol. 1,2003
7,27,9 Songs,2004
8,28,Apocalypse Now,1979
9,33,Unforgiven,1992


In [9]:
# Page 2 : on doit obtenir 10 films différents de la page 1
reponse = client.get("/movie/", params={"page": 2})
pd.DataFrame(reponse.json()["resultats"])

,id,title,annee
0,35,The Simpsons Movie,2007
1,38,Eternal Sunshine of the Spotless Mind,2004
2,55,Amores Perros,2000
3,58,Pirates of the Caribbean: Dead Man's Chest,2006
4,62,2001: A Space Odyssey,1968
5,63,Twelve Monkeys,1995
6,65,8 Mile,2002
7,70,Million Dollar Baby,2004
8,73,American History X,1998
9,74,War of the Worlds,2005


## 3. Détail d'un film

On récupère un id valide depuis la page 1, puis on regarde le détail complet.

In [6]:
premier_film = client.get("/movie/").json()["resultats"][0]
film_id = premier_film["id"]
print(f"Film choisi : {premier_film['title']} (id {film_id})")

detail = client.get(f"/movie/{film_id}").json()
print("Genres :", detail["genres"])
print("Tags :", detail["tags"])
pd.DataFrame(detail["equipe"])

Film choisi : Finding Nemo (id 12)
Genres : ['Adventure', 'Family', 'Animation']
Tags : ['disney', 'fish', 'heartwarming']


,id,name,role,personnage
0,12,Alexander Gould,Acteur,Nemo (voice)
1,13,Albert Brooks,Acteur,Marlin (voice)
2,14,Ellen DeGeneres,Acteur,Dory (voice)
3,118,Geoffrey Rush,Acteur,Nigel (voice)
4,5293,Willem Dafoe,Acteur,Gill (voice)


### Cas d'erreur : film inexistant

In [10]:
reponse = client.get("/movie/999999999")
print(reponse.status_code)
reponse.json()

404


{'detail': 'Film 999999999 non trouvé'}

### Cas d'erreur : identifiant mal typé

FastAPI valide le type avant même d'exécuter la fonction de la route.

In [11]:
reponse = client.get("/movie/abc")
print(reponse.status_code)
reponse.json()

422


{'detail': [{'type': 'int_parsing',
   'loc': ['path', 'film_id'],
   'msg': 'Input should be a valid integer, unable to parse string as an integer',
   'input': 'abc'}]}

## 4. Classements

In [9]:
pd.DataFrame(client.get("/top_actor/").json())

,id,name,nombre_films
0,3223,Robert Downey Jr.,8
1,31,Tom Hanks,5
2,1813,Anne Hathaway,5
3,10980,Daniel Radcliffe,5
4,10989,Rupert Grint,5
5,10990,Emma Watson,5
6,15277,Jon Favreau,5
7,16828,Chris Evans,5
8,103,Mark Ruffalo,4
9,287,Brad Pitt,4


In [10]:
pd.DataFrame(client.get("/top_director/").json())

,id,name,nombre_films
0,525,Christopher Nolan,6
1,2710,James Cameron,5
2,108,Peter Jackson,4
3,138,Quentin Tarantino,3
4,578,Ridley Scott,3
5,1776,Francis Ford Coppola,3
6,19271,Anthony Russo,3
7,19272,Joe Russo,3
8,137427,Denis Villeneuve,3
9,24,Robert Zemeckis,2


In [11]:
pd.DataFrame(client.get("/top_revenue/").json())

,id,title,revenue
0,19995,Avatar,2923706026
1,299534,Avengers: Endgame,2799439100
2,969681,Spider-Man: Brand New Day,2478502286
3,76600,Avatar: The Way of Water,2334484620
4,597,Titanic,2264162353
5,299536,Avengers: Infinity War,2052415039
6,634649,Spider-Man: No Way Home,1921426073
7,1084242,Zootopia 2,1868208796
8,1368337,The Odyssey,1723918419
9,24428,The Avengers,1518815515


In [12]:
pd.DataFrame(client.get("/top_budget/").json())

,id,title,budget
0,76600,Avatar: The Way of Water,460000000
1,575265,Mission: Impossible - The Final Reckoning,400000000
2,299534,Avengers: Endgame,356000000
3,83533,Avatar: Fire and Ash,350000000
4,385687,Fast X,340000000
5,299536,Avengers: Infinity War,300000000
6,911430,F1,300000000
7,49026,The Dark Knight Rises,250000000
8,122917,The Hobbit: The Battle of the Five Armies,250000000
9,447365,Guardians of the Galaxy Vol. 3,250000000


## 5. Pydantic valide vraiment, pas seulement pour la doc

On construit un modèle directement, avec un type invalide, pour voir l'erreur de validation.

In [13]:
from pydantic import ValidationError
from schemas import FilmResume

try:
    FilmResume(id="pas-un-nombre", title="Test")
except ValidationError as erreur:
    print(erreur)

1 validation error for FilmResume
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='pas-un-nombre', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


---
**À vous :** ouvrez `http://127.0.0.1:8000/docs` (après avoir lancé `uvicorn main:app --reload --app-dir api` dans un terminal) pour retrouver ces mêmes routes dans Swagger, organisées par les 3 tags. Le `TestClient` utilisé ici est pratique pour explorer vite, mais Swagger reste la référence pour tester manuellement et partager la documentation.